# Aligned AR on short supports with only 2 empty slots (seed 42)

Two settings in one notebook (run one after the other):
* **5 slots, trained k = 0, 2, 3**: tests single operations and k = 4, 5;
* **4 slots, trained k = 0, 2**: tests single operations and k = 3, 4.

**Question.** Aligned AR (no sequence position, target-cell coordinates, 10-way digit head) learns single
operations whenever every training instruction leaves at most 2 slots empty. So far, however, it has only been
tested with a longest training instruction of 4 or more. Here the instructions are short (longest 3 or 2) but still
leave only 2 slots empty.
* **Success** → the number of empty slots is what matters.
* **Failure** → a short longest instruction is itself a problem.

Standard AR failed on 5 slots / k ≤ 3 (0/15000 singletons, earlier experiments).

**Setup.** Same model, data stream, 20,000 updates and sampled LAST-checkpoint evaluation as notebooks 34 / 36 / 39.
The last cell shows accuracy by k and the failure analysis for both settings.
**Send back `AlignedAR_gap2_short_Review.ipynb`.**


In [1]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


Repository: /content/Decompositionality | commit: ba6f54a898211af0c52c93cbf1b48ba003d41280


In [2]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = os.environ.get("DECOMP_MODE", "full")
SEED = 42
RUN_CONDITIONS = ['S5_k023', 'S4_k02']
RUN_MODELS = ["ar_slotpos_digithead_target"]
STEPS = 20000
SUBDIR = "S7_support_sweep_20k_v1"                # same folder as notebooks 23 / 27 (model subfolders differ)
EVAL_PLAN = {"n_single_grids": 1000, "n_seen_per_k": 5000, "n_long_per_k": 20000}
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import longk as LK, longk_exp as LX
print("results ->", LX.family_root(RESULTS_ROOT, MODE, SUBDIR))


Mounted at /content/drive
results -> /content/drive/MyDrive/decompositionality_results/S7_support_sweep_20k_v1


In [3]:
# --- Train (20,000 updates) + sampled LAST-checkpoint evaluation (resumable) ---------------------------------------
status = LX.run_queue(RESULTS_ROOT, MODE, SEED, RUN_CONDITIONS, RUN_MODELS, repo_dir=str(REPO_DIR), device=DEVICE,
                      subdir=SUBDIR, steps=STEPS if MODE == "full" else None, sampled_plan=EVAL_PLAN)
display(status)


[S5_k023] S=5 train k=[0, 2, 3] | val 900 | manifests bdf24c327a65

=== S5_k023 / ar_slotpos_digithead_target ===
  microbatch probe: 256 fits
[ar_slotpos_digithead_target] 10,670,218 params | device NVIDIA RTX PRO 6000 Blackwell Server Edition | dtype bf16 | microbatch 256 x accumulation 1 = 256
  [val @ 250] cell CE 1.2477 (best) | subset(96) exact 0.323 cell 0.495
  [val @ 500] cell CE 0.5407 (best) | subset(96) exact 0.406 cell 0.751
  step 600/20000 loss 0.2523 lr 3.00e-04 | 28.86 steps/s, 7389 ex/s | ETA 11.2 min
  [val @ 750] cell CE 0.0647 (best) | subset(96) exact 0.667 cell 0.973
  [val @ 1000] cell CE 0.0073 (best) | subset(96) exact 0.958 cell 0.999
  step 1200/20000 loss 0.0021 lr 2.98e-04 | 29.09 steps/s, 7448 ex/s | ETA 10.8 min
  [val @ 1250] cell CE 0.0051 (best) | subset(96) exact 0.958 cell 0.998
  [val @ 1500] cell CE 0.0040 (best) | subset(96) exact 0.990 cell 0.999
  step 1750/20000 loss 0.0035 lr 2.96e-04 | 28.46 steps/s, 7285 ex/s | ETA 10.7 min
  [val @ 1750] c

,condition,model,trained,evaluated_last,records,best_step (reference),complete
0,S5_k023,ar_slotpos_digithead_target,True,True,66000,10250,True
1,S4_k02,ar_slotpos_digithead_target,True,True,58000,4750,True


In [4]:
# --- Accuracy and outcome analysis for both settings  ------------------------------
from decompositionality import shortcut_analysis as SA
conds = LK.GAP2_SHORT
df = LX.per_k_accuracy(RESULTS_ROOT, MODE, [SEED], conds, SUBDIR, RUN_MODELS)
ok = df[df.status == "available"]
print("Exact accuracy (rows: setting, columns: test k; k = 1 = single operations):")
display(ok.pivot_table(index="condition", columns="k", values="exact_acc").round(4))
rp = SA.analyse(RESULTS_ROOT, MODE, SEED, conditions=conds, models=RUN_MODELS,
                review_name="AlignedAR_gap2_short_Review.ipynb", out_name=f"seed_{SEED}_aligned_gap2_short")
print("REVIEW FILE TO SEND BACK:", rp)
if IN_COLAB and rp:
    from google.colab import files
    files.download(str(rp))


Exact accuracy (rows: setting, columns: test k; k = 1 = single operations):


k,0,1,2,3,4,5
condition,,,,,,
S4_k02,1.0,0.0,1.0,0.0,0.062,NaN
S5_k023,1.0,0.0,1.0,1.0,0.000,0.0019


# What do the models compute on held-out inputs? (seed 42, LAST checkpoint)
Read-only: rebuilt sampled test sets are checked record-by-record against the saved predictions. A transform is written as its shortest A/B/C word (A = right shift, B = clockwise rotation, C = left-right mirror; executed left to right). 'seen' = the transform is the overall transform of some trained instruction.


### Availability


,condition,model,status
0,S5_k023,AR34,ok
1,S4_k02,AR34,ok


**Which operation is dropped?** For held-out longer instructions whose prediction equals the word with exactly ONE operation removed, the position (1 = first, k = last) of that operation. 'unique' counts only records where exactly one position fits; 'any' counts every fitting position.

... (11 outputs left out)

,condition,model,k,records,correct,identity copy,= one op dropped,of which = last op dropped,other legal transform,no legal / invalid
0,S5_k023,AR34,4,20000,0.0,0.000,0.382,0.143,0.382,0.618
1,S4_k02,AR34,3,20000,0.0,0.038,0.631,0.333,0.593,0.369


REVIEW FILE: /content/drive/MyDrive/decompositionality_results/S7_support_sweep_20k_v1/report/seed_42_aligned_gap2_short/AlignedAR_gap2_short_Review.ipynb
REVIEW FILE TO SEND BACK: /content/drive/MyDrive/decompositionality_results/S7_support_sweep_20k_v1/report/seed_42_aligned_gap2_short/AlignedAR_gap2_short_Review.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>